In [2]:
import pandas as pd
import numpy as np
#1. UPLOAD DATASET
from google.colab import files
uploaded = files.upload()


Saving SurveilDrone-Net23.csv to SurveilDrone-Net23.csv


In [3]:
#2. LOAD DATA
df = pd.read_csv("SurveilDrone-Net23.csv")

print(df.shape)

df.head()

(140256, 34)


,timestamp,mission_id,drone_id,altitude_m,velocity_x,velocity_y,velocity_z,acceleration_x,acceleration_y,acceleration_z,...,wind_speed_mps,wind_dir_deg,weather_condition,region_type,proximity_to_restricted_zone_m,detected_object_count,average_object_size_px,thermal_signature_intensity,detection_confidence_avg,surveillance_pattern
0,2021-01-01 00:00:00,MSN2824,DR2,56,-3.003620,1.745126,-1.012630,-0.116718,0.654640,0.591926,...,2.273547,240,Cloudy,Urban,30,2,45.220770,107.928854,0.577466,Patrol
1,2021-01-01 00:15:00,MSN1409,DR3,112,0.898677,-6.052608,0.084181,-0.491444,-1.160391,-0.987538,...,1.756862,218,Clear,Forest,340,2,93.857320,146.794767,0.726129,Patrol
2,2021-01-01 00:30:00,MSN5506,DR6,87,-1.642041,0.605855,-2.683689,0.410175,-1.378734,-0.958764,...,4.562560,150,Cloudy,Coastal,69,1,68.653391,109.341661,0.810754,Circle
3,2021-01-01 00:45:00,MSN5012,DR7,74,0.041976,-2.872027,0.718481,0.454186,-0.609361,-1.890276,...,1.735225,88,Clear,Urban,71,0,77.802779,157.431678,0.512223,Scan
4,2021-01-01 01:00:00,MSN4657,DR6,40,6.469317,0.043959,0.361757,-0.710856,-2.772141,0.371007,...,0.389182,180,Clear,Forest,316,2,51.667683,79.343379,0.878731,Scan


## Dataset Overview

The dataset contains UAV telemetry records including battery status,
GPS location, mission information, weather conditions,
and surveillance-related variables.

In [4]:
#3. DATASET OVERVIEW
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140256 entries, 0 to 140255
Data columns (total 34 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   timestamp                       140256 non-null  object 
 1   mission_id                      140256 non-null  object 
 2   drone_id                        140256 non-null  object 
 3   altitude_m                      140256 non-null  int64  
 4   velocity_x                      140256 non-null  float64
 5   velocity_y                      140256 non-null  float64
 6   velocity_z                      140256 non-null  float64
 7   acceleration_x                  140256 non-null  float64
 8   acceleration_y                  140256 non-null  float64
 9   acceleration_z                  140256 non-null  float64
 10  heading_deg                     140256 non-null  int64  
 11  gps_lat                         140256 non-null  float64
 12  gps_lon         

## Missing Value Analysis

This step identifies variables containing missing values
before cleaning.

In [5]:
#4. CHECK MISSING VALUES
df.isnull().sum()

,0
timestamp,0
mission_id,0
drone_id,0
altitude_m,0
velocity_x,0
velocity_y,0
velocity_z,0
acceleration_x,0
acceleration_y,0
acceleration_z,0


## Duplicate Record Analysis

In [6]:
df.duplicated().sum()

np.int64(0)

## Data Type Validation

In [8]:
df.dtypes

,0
timestamp,object
mission_id,object
drone_id,object
altitude_m,int64
velocity_x,float64
velocity_y,float64
velocity_z,float64
acceleration_x,float64
acceleration_y,float64
acceleration_z,float64


#DATA CLEANING

In [12]:
#1. Covert Timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"])

#2. Fix Wind Speed
negative_wind = (df["wind_speed_mps"] < 0).sum()
print("Negative wind records: ", negative_wind)
df["wind_speed_mps"] = abs(df["wind_speed_mps"])

Negative wind records:  3171


## Wind Speed Validation

Negative wind speed values were converted to absolute values.

In [13]:
#3.Check Battery Range
df["battery_level_pct"].describe()

,battery_level_pct
count,140256.000000
mean,39.238407
std,24.361640
min,5.000000
25%,19.000000
50%,35.000000
75%,57.000000
max,100.000000


In [14]:
#4. Check GPS Range
df["gps_lat"].describe()

df["gps_lon"].describe()

,gps_lon
count,140256.000000
mean,72.999419
std,0.577569
min,72.000001
25%,72.500411
50%,72.998054
75%,73.499283
max,73.999999


In [15]:
#5.Remove Duplicates
df = df.drop_duplicates()

##DSS FEATURE ENGINEERING

In [17]:
df["speed_mps"] = np.sqrt(
    df["velocity_x"]**2 +
    df["velocity_y"]**2 +
    df["velocity_z"]**2
)

In [18]:
# Create Battery Risk
def battery_risk(x):

    if x < 15:
        return "Critical"

    elif x < 30:
        return "High"

    elif x < 50:
        return "Medium"

    else:
        return "Low"

df["battery_risk_level"] = df["battery_level_pct"].apply(
    battery_risk
)

In [19]:
# Create Weather Risk
def weather_risk(x):

    if x >= 10:
        return "High"

    elif x >= 6:
        return "Medium"

    else:
        return "Low"

df["weather_risk_level"] = df["wind_speed_mps"].apply(
    weather_risk
)

In [20]:
def recommend(row):

    if row["battery_level_pct"] < 15:
        return "Return to Base"

    elif row["wind_speed_mps"] > 10:
        return "Delay Mission"

    else:
        return "Continue Mission"

df["recommended_action"] = df.apply(
    recommend,
    axis=1
)

In [21]:
# EXPORT CLEANED DATASET
df.to_csv(
    "SurveilDrone_Net23_DSS_cleaned.csv",
    index=False
)

In [22]:
#Download
from google.colab import files

files.download(
    "SurveilDrone_Net23_DSS_cleaned.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>